In [ ]:
# ============================================================
# MASK R-CNN (SCRATCH) + JSON SPLIT + OOM FIXED VERSION
# FULL PIPELINE: DATASET → TRAIN → EVALUATE
# ============================================================

# ================= IMPORTS =================
import os, json, torch, numpy as np
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader

# ============================================================
# LOAD JSON SPLIT
# ============================================================
SPLIT_JSON_PATH = "/kaggle/input/datasets/iharshsinha/splitjson/data_split.json"

with open(SPLIT_JSON_PATH) as f:
    split_data = json.load(f)

train_files = [f.replace(".jpg", ".json") for f in split_data["train"]]
val_files   = [f.replace(".jpg", ".json") for f in split_data["val"]]

print("Train samples:", len(train_files))
print("Val samples:", len(val_files))

# ============================================================
# DATASET CLASS (WITH RESIZE FOR MEMORY FIX)
# ============================================================
class MaskRCNNDataset(Dataset):

    def __init__(self, images_dir, annos_dir, file_list):
        self.images_dir = images_dir
        self.annos_dir = annos_dir
        self.files = file_list

        self.label_map = {
            "short sleeve top": 1,
            "trousers": 2,
            "shorts": 3,
            "long sleeve top": 4,
            "skirt": 5
        }

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        file = self.files[idx]

        json_path = os.path.join(self.annos_dir, file)
        img_path = os.path.join(self.images_dir, file.replace(".json", ".jpg"))

        # -------- HANDLE BAD IMAGES --------
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            return self.__getitem__((idx + 1) % len(self))

        # 🔥 RESIZE IMAGE (VERY IMPORTANT FOR GPU)
        image = image.resize((512, 512))
        w, h = image.size

        with open(json_path) as f:
            data = json.load(f)

        boxes, labels, masks = [], [], []

        for key, val in data.items():

            if not key.startswith("item"):
                continue

            category = val["category_name"]

            if category not in self.label_map:
                continue

            boxes.append(val["bounding_box"])
            labels.append(self.label_map[category])

            # -------- MASK --------
            mask = Image.new("L", (w, h), 0)

            for polygon in val["segmentation"]:
                if len(polygon) >= 6:
                    ImageDraw.Draw(mask).polygon(polygon, outline=1, fill=1)

            # 🔥 RESIZE MASK
            mask = mask.resize((512, 512))
            masks.append(np.array(mask, dtype=np.uint8))

        if len(boxes) == 0:
            return self.__getitem__((idx + 1) % len(self))

        # -------- IMAGE FIX --------
        image = np.array(image)

        if image.ndim == 2:
            image = np.stack([image]*3, axis=-1)

        if image.shape[2] != 3:
            image = image[:, :, :3]

        image = torch.as_tensor(image.transpose(2,0,1), dtype=torch.float32) / 255.0

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": torch.as_tensor(np.array(masks), dtype=torch.uint8),
            "image_id": torch.tensor([idx])
        }

        return image, target

# ============================================================
# PATHS
# ============================================================
BASE_PATH = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed"

IMAGES = os.path.join(BASE_PATH,"train/images")
ANNOS  = os.path.join(BASE_PATH,"train/annos")

# ============================================================
# DATASETS
# ============================================================
train_dataset = MaskRCNNDataset(IMAGES, ANNOS, train_files)
val_dataset   = MaskRCNNDataset(IMAGES, ANNOS, val_files)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

# ============================================================
# DATALOADER (REDUCED BATCH)
# ============================================================
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)

# ============================================================
# MODEL (SCRATCH)
# ============================================================
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 6

model = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)

# heads
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)

model.to(device)

print("Model initialized FROM SCRATCH")

# ============================================================
# TRAINABLE
# ============================================================
for param in model.parameters():
    param.requires_grad = True

# ============================================================
# OPTIMIZER
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ============================================================
# TRAIN FUNCTION (AMP + GRAD ACCUMULATION)
# ============================================================
scaler = torch.cuda.amp.GradScaler()
accum_steps = 2

def train_one_epoch(model, loader):

    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for i, (images, targets) in enumerate(loader):

        images = [img.to(device) for img in images]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

        with torch.cuda.amp.autocast():
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            losses = losses / accum_steps

        scaler.scale(losses).backward()

        if (i + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += losses.item()

    return total_loss / len(loader)

# ============================================================
# TRAIN LOOP
# ============================================================
epochs = 5
best_loss = float("inf")

for epoch in range(epochs):

    print("\n========================")
    print("Epoch", epoch+1)
    print("========================")

    loss = train_one_epoch(model, train_loader)

    print("Training Loss:", loss)

    if loss < best_loss:
        best_loss = loss
        torch.save(model.state_dict(), "/kaggle/working/maskrcnn_scratch_best.pth")
        print("Best model saved")

print("\nTraining Complete")
print("Best Loss:", best_loss)

# ============================================================
# METRICS
# ============================================================
def compute_iou(box1, box2):
    x1=max(box1[0],box2[0]); y1=max(box1[1],box2[1])
    x2=min(box1[2],box2[2]); y2=min(box1[3],box2[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    a1=(box1[2]-box1[0])*(box1[3]-box1[1])
    a2=(box2[2]-box2[0])*(box2[3]-box2[1])
    return inter/(a1+a2-inter+1e-6)

def compute_mask_metrics(pred,gt):
    inter=np.logical_and(pred,gt).sum()
    union=np.logical_or(pred,gt).sum()
    iou=inter/(union+1e-6)
    dice=2*inter/(pred.sum()+gt.sum()+1e-6)
    return iou,dice

# ============================================================
# EVALUATION
# ============================================================
def evaluate_model(model, loader):

    model.eval()
    total_iou=[]; total_dice=[]
    correct=0; total=0

    with torch.no_grad():

        for images, targets in loader:

            images = [img.to(device) for img in images]
            outputs = model(images)

            for i in range(len(outputs)):

                preds = outputs[i]
                gt = targets[i]

                for pb,pl in zip(preds["boxes"].cpu(), preds["labels"].cpu()):
                    best=0
                    for gb,gl in zip(gt["boxes"], gt["labels"]):
                        if pl!=gl: continue
                        best=max(best,compute_iou(pb.numpy(),gb.numpy()))
                    if best>0.5: correct+=1
                    total+=1

                for pm,gm in zip(preds["masks"], gt["masks"]):
                    pm=(pm[0]>0.5).cpu().numpy()
                    gm=gm.numpy()
                    iou,dice=compute_mask_metrics(pm,gm)
                    total_iou.append(iou)
                    total_dice.append(dice)

    precision = correct/total if total>0 else 0
    mIoU = np.mean(total_iou)
    dice = np.mean(total_dice)

    print("\n========================")
    print("EVALUATION RESULTS (SCRATCH)")
    print("========================")
    print("Precision:",precision)
    print("mIoU:",mIoU)
    print("Dice:",dice)

    return precision,mIoU,dice

# ================= RUN EVAL =================
evaluate_model(model, val_loader)